# L3.4 · ACT: el entrenamiento por dentro


### El mapa de toda la serie

ACT es un **VAE condicional**:

- **Encoder del VAE** (sólo al entrenar): lee el estado y el bloque de acciones
  verdadero y lo resume en una variable `z`, el "estilo" de esa demostración.
- **Decoder = la policy** (al entrenar y al usar): un transformer
  encoder-decoder al estilo DETR. Lee la cámara, el estado y la `z`, y escribe
  el bloque entero de acciones de una pasada.
- **Al usarlo:** el encoder del VAE se tira y `z = 0`.

| Notebook | Qué abre |
|---|---|
| L3.1 | Los datos: de la grabación al ejemplo de entrenamiento |
| L3.2 | El encoder del VAE y la `z` |
| L3.3 | El decoder: la policy por dentro |
| L3.4 | El entrenamiento: un paso desarmado, y el modelo aprendiendo |
| L3.5 | El modelo en uso: ejecutar los bloques |

Todo usa el código real de LeRobot 0.4.1 y **nuestro modelo entrenado**
(`act_cheap_v1`), desarmado con `toolkit/policy/act_internals.py`. Sus tests
verifican que las piezas encadenadas dan exactamente lo mismo que el modelo.

In [ ]:
import torch  # antes que lerobot: evita un import circular entre torchvision y torch._dynamo
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from lerobot.datasets.lerobot_dataset import LeRobotDataset

from toolkit.paths import CHECKPOINTS_DIR, DATASETS_DIR, REPO_ROOT
from toolkit.policy import act_internals as act
from toolkit.policy.load import load_policy

plt.rcParams["figure.dpi"] = 110
torch.manual_seed(0)
ROOT = DATASETS_DIR / "local" / "pick_mameluco_act_v1"
CAMERA = "observation.images.cam_right_high"
loaded = load_policy(CHECKPOINTS_DIR / "act_cheap_v1" / "pretrained_model", device="cpu")
policy, model, config = loaded.policy, loaded.policy.model, loaded.policy.config
_ = policy.eval()

Dos partes:

1. **Un paso desarmado**, con nuestro modelo ya entrenado: qué error calcula,
   cuánto pesa cada término, y por dónde viajan los gradientes.
2. **Un ACT desde cero**, entrenado acá unos minutos, con "fotos" en el camino:
   cómo predice recién inicializado y cómo va aprendiendo.

## 1. Un paso desarmado

### El error, término por término

```latex
\mathcal{L} = \underbrace{\operatorname{mean}\big(|a - \hat a| \cdot \text{no relleno}\big)}_{\text{L1: reconstrucción}} + \beta \cdot \underbrace{\operatorname{mean}_{\text{lote}}\textstyle\sum_{d} \mathrm{KL}_d}_{\text{KL}}
```

con β = `kl_weight` = 10. El paper eligió **L1** (error absoluto) en vez del
cuadrático porque daba acciones más precisas: el cuadrático castiga mucho los
errores grandes y "aplana" la respuesta hacia el promedio.

In [ ]:
from lerobot.datasets.factory import resolve_delta_timestamps
from lerobot.datasets.lerobot_dataset import LeRobotDatasetMetadata

meta = LeRobotDatasetMetadata("local/pick_mameluco_act_v1", root=ROOT)
dataset = LeRobotDataset("local/pick_mameluco_act_v1", root=ROOT, video_backend="pyav",
                         delta_timestamps=resolve_delta_timestamps(config, meta))
loader = torch.utils.data.DataLoader(dataset, batch_size=16, shuffle=True, generator=torch.Generator().manual_seed(0))
batch = loaded.preprocessor(next(iter(loader)))

policy.train()
loss, parts = policy.forward(batch)
print(f"pérdida {loss.item():.4f} = L1 {parts['l1_loss']:.4f} + {config.kl_weight} × KL {parts['kld_loss']:.6f}")

En nuestro modelo el KL es prácticamente cero: el VAE colapsó (L3.2). Toda la
pérdida es reconstrucción.

### ¿Dónde se equivoca más? Por paso futuro y por articulación

In [ ]:
policy.eval()
with torch.no_grad():
    style = act.encode_style(model, batch)
    scene = act.scene_tokens(model, batch, act.sample_latent(style))
    _, predicted = act.decode_chunk(model, act.encode_scene(model, scene), scene)
error = (predicted - batch["action"]).abs() * ~batch["action_is_pad"].unsqueeze(-1)
names = meta.features["action"]["names"][0]
fig, axes = plt.subplots(1, 2, figsize=(15, 3.2))
axes[0].plot(error.mean((0, 2))); axes[0].set_xlabel("paso del bloque"); axes[0].set_title("error L1 normalizado por paso futuro")
axes[1].bar(names, error.mean((0, 1))); axes[1].set_title("por articulación"); axes[1].tick_params(axis="x", rotation=90, labelsize=7)
plt.tight_layout()

El error crece con el horizonte: predecir lo que viene en 1,5 s es más difícil
que lo que viene ya. Pero cada paso pesa lo mismo en la pérdida.

### Por dónde viajan los gradientes

`backward` calcula, para cada peso, cuánto cambia el error si se lo mueve. Lo
miramos por pieza: eso es lo que se va a ajustar en este paso.

In [ ]:
policy.train()
policy.zero_grad()
loss, _ = policy.forward(batch)
loss.backward()
pieces = {"ResNet18": model.backbone, "encoder del VAE": model.vae_encoder, "encoder": model.encoder,
          "decoder": model.decoder, "cabeza": model.action_head}
norms = {name: torch.sqrt(sum(p.grad.pow(2).sum() for p in m.parameters() if p.grad is not None)).item() for name, m in pieces.items()}
total = torch.sqrt(sum(p.grad.pow(2).sum() for p in policy.parameters() if p.grad is not None)).item()
policy.zero_grad(); policy.eval()
print(f"norma total del gradiente: {total:.3f} (se recorta a {10.0})")
pd.Series(norms).round(4).to_frame("norma del gradiente")

El encoder del VAE sigue recibiendo gradiente. ¿De dónde, si la `z` no se
usa? Separamos la pérdida en sus dos términos y miramos qué le llega a él de
cada uno:

In [ ]:
def vae_encoder_gradient(term):
    policy.train(); policy.zero_grad()
    torch.manual_seed(3)  # el mismo sorteo de z para los dos términos
    full = dict(batch, **{"observation.images": [batch[k] for k in config.image_features]})
    predicted, (mu, log_sigma_x2) = model(full)
    l1 = ((predicted - batch["action"]).abs() * ~batch["action_is_pad"].unsqueeze(-1)).mean()
    kl = (-0.5 * (1 + log_sigma_x2 - mu.pow(2) - log_sigma_x2.exp())).sum(-1).mean()
    (l1 if term == "L1" else config.kl_weight * kl).backward()
    norm = torch.sqrt(sum(p.grad.pow(2).sum() for p in model.vae_encoder.parameters() if p.grad is not None)).item()
    policy.zero_grad(); policy.eval()
    return norm

print(f"gradiente al encoder del VAE · desde L1: {vae_encoder_gradient('L1'):.4f} · desde {config.kl_weight} × KL: {vae_encoder_gradient('KL'):.4f}")

Casi todo viene del **KL**. La reconstrucción ya no le pide nada a la `z`
(el decoder la ignora), y lo único que sigue empujando al encoder es el KL,
para mantenerlo pegado a la campana estándar. Es el colapso visto desde los
gradientes.

El optimizador (AdamW) usa además **dos tasas de aprendizaje**: una baja para
la ResNet, que ya sabe ver, y otra para el resto.

## 2. Un ACT desde cero, con fotos en el camino

Entrenamos un ACT nuevo, con la misma configuración que el nuestro, sólo con
los episodios de entrenamiento, y lo fotografiamos en los pasos 0, 25, 100,
300, 1000 y 2000. En cada foto, le pedimos el bloque para **un cuadro de un
episodio que nunca ve**.

Tarda unos 15 minutos en la GPU de la Mac. Con `STEPS` más chico se hace más
corto.

In [ ]:
from lerobot.configs.types import FeatureType
from lerobot.datasets.factory import IMAGENET_STATS
from lerobot.datasets.utils import dataset_to_policy_features
from lerobot.policies.act.configuration_act import ACTConfig
from lerobot.policies.act.modeling_act import ACTPolicy
from lerobot.policies.factory import make_pre_post_processors

DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"
SNAPSHOTS = [0, 25, 100, 300, 1000, 2000]
split = json.loads((REPO_ROOT / "learning" / "act" / "split.json").read_text())

features = dataset_to_policy_features(meta.features)
outputs = {k: v for k, v in features.items() if v.type is FeatureType.ACTION}
inputs = {k: v for k, v in features.items() if k not in outputs}
fresh_config = ACTConfig(input_features=inputs, output_features=outputs, device=DEVICE,
                         chunk_size=48, n_action_steps=48, dim_model=192, dim_feedforward=768, n_heads=4,
                         optimizer_lr=5e-5)

stats = {k: dict(v) for k, v in meta.stats.items()}
for key, value in IMAGENET_STATS.items():
    stats[CAMERA][key] = torch.tensor(value, dtype=torch.float32)
pre, post = make_pre_post_processors(fresh_config, dataset_stats=stats)

torch.manual_seed(0)
fresh = ACTPolicy(fresh_config).to(DEVICE)
optimizer = fresh_config.get_optimizer_preset().build(fresh.get_optim_params())

# Sólo los cuadros de los episodios de entrenamiento (episodes= no filtra en LeRobot 0.4.1).
train_frames = [i for e in split["train_episode_indices"]
                for i in range(int(meta.episodes[e]["dataset_from_index"]), int(meta.episodes[e]["dataset_to_index"]))]
train_loader = torch.utils.data.DataLoader(dataset, batch_size=8, num_workers=0, drop_last=True,
                                           sampler=torch.utils.data.SubsetRandomSampler(train_frames))

held_episode = split["heldout_episode_indices"][1]
probe_index = int(meta.episodes[held_episode]["dataset_from_index"]) + 120
probe = dataset[probe_index]
probe_obs = pre({k: probe[k] for k in ["observation.state", CAMERA]})
print(len(train_frames), "cuadros de entrenamiento · cuadro de prueba del episodio no visto", held_episode)

El ciclo es el de `lerobot-train` (L5), más una foto en los pasos elegidos:

In [ ]:
import time

log, snapshots = [], {}
batches = iter(train_loader)
start_time = time.time()
for step in range(SNAPSHOTS[-1] + 1):
    if step in SNAPSHOTS:
        fresh.eval()
        with torch.no_grad():
            snapshots[step] = post(fresh.predict_action_chunk(probe_obs))[0].cpu().numpy()
        print(f"foto en el paso {step:5d} · {time.time() - start_time:5.0f} s")
    if step == SNAPSHOTS[-1]:
        break
    try:
        raw = next(batches)
    except StopIteration:
        batches = iter(train_loader)
        raw = next(batches)
    train_batch = pre(raw)
    fresh.train()
    loss, parts = fresh.forward(train_batch)
    loss.backward()
    grad_norm = torch.nn.utils.clip_grad_norm_(fresh.parameters(), 10.0)
    optimizer.step(); optimizer.zero_grad()
    if step % 10 == 0:
        with torch.no_grad():
            style = act.encode_style(fresh.model, train_batch)
        log.append({"paso": step, "L1": parts["l1_loss"], "KL": parts["kld_loss"], "gradiente": float(grad_norm),
                    "|μ| medio": float(style.mu.abs().mean()), "σ medio": float(style.sigma.mean())})
log = pd.DataFrame(log).set_index("paso")

### Cómo baja el error, y qué le pasa a la `z`

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(17, 3))
for ax, column in zip(axes, ["L1", "KL", "|μ| medio", "σ medio"]):
    ax.plot(log.index, log[column], alpha=0.4); ax.plot(log.index, log[column].rolling(10, min_periods=1).mean(), lw=2)
    ax.set_title(column); ax.set_xlabel("paso")
axes[1].set_yscale("log")
plt.tight_layout()
log.iloc[[0, 2, 10, 30, 100, len(log) - 1]].round(4)

### Cómo va aprendiendo: el mismo cuadro no visto, en cada foto

In [ ]:
names = meta.features["action"]["names"][0]
truth = probe["action"].numpy()
fig, axes = plt.subplots(1, 3, figsize=(16, 3.6))
colors = plt.cm.viridis(np.linspace(0, 1, len(SNAPSHOTS)))
for ax, joint in zip(axes, ["kRightShoulderPitch", "kRightElbow", "kRightHandIndex"]):
    j = names.index(joint)
    ax.plot(truth[:, j], color="k", lw=3, label="operador")
    for color, step in zip(colors, SNAPSHOTS):
        ax.plot(snapshots[step][:, j], color=color, label=f"paso {step}")
    ax.set_title(joint); ax.set_xlabel("paso del bloque")
axes[0].legend(fontsize=7)
plt.tight_layout()

for step in SNAPSHOTS:
    print(f"paso {step:5d}: error medio {np.abs(snapshots[step] - truth).mean():.3f} rad")

Para leer este gráfico, mirá el orden en que aparecen las cosas. Lo que
sigue es lo que vimos en nuestra corrida de prueba; la tuya puede variar un
poco en los números, no en el orden:

1. **Paso 0**: la red está inicializada al azar. La cabeza lineal devuelve
   valores del orden de ±1 en la escala normalizada; des-normalizado, es un
   bloque **cerca de la acción promedio del dataset** (en la prueba, a 0,4
   desvíos en promedio), corrido al azar y **casi plano**, sin relación con la
   imagen. Error: unos 0,25 rad.
2. **Primeros cientos de pasos**: aprende el **nivel**, dónde está el brazo
   ahora. El error baja rápido (en la prueba, 0,08 rad al paso 25 y 0,05 al
   300), pero los bloques siguen planos: el codo variaba 0,02 rad dentro del
   bloque cuando el operador lo movió 0,41. Es lo más fácil, y equivale a la
   estrategia "quedarse quieto".
3. **Hacia el paso 1000**: aparece la **forma**, hacia dónde va el movimiento
   dentro del bloque (el codo ya variaba 0,24 rad). Eso es lo que le gana a
   "quedarse quieto".

Ojo: el error de **un** cuadro sube y baja de una foto a otra; la tendencia se
ve en la curva de L1, que promedia muchos lotes.

Nuestro modelo entrenado tuvo 30.000 pasos con lotes de 64: unas 120 veces
más ejemplos vistos que esta corrida.

**La `z` colapsa de a poco, no de golpe.** Al principio el encoder del VAE
devuelve campanas distintas para cada bloque. En la prueba el KL arrancó en
9,5, bajó a 0,35 al paso 100 y a 0,02 al paso 2000. σ llega a 1 en las
primeras decenas de pasos; lo que se apaga lentamente es μ (de 0,43 a 0,02).
El KL × 10 va apagando la `z` mientras el decoder aprende a no necesitarla.

**El gradiente se recorta todo el tiempo.** En la prueba su norma arrancó en
664 y terminó en 18: siempre por encima del límite de 10. Al principio del
entrenamiento, el recorte es lo que evita que un paso rompa todo.

### Comparar con el modelo entrenado

In [ ]:
with torch.no_grad():
    trained = loaded.postprocessor(policy.predict_action_chunk(loaded.preprocessor({k: probe[k] for k in ["observation.state", CAMERA]})))[0].numpy()
print(f"modelo entrenado (30.000 pasos): error medio {np.abs(trained - truth).mean():.3f} rad")
print(f"este, tras {SNAPSHOTS[-1]} pasos           : error medio {np.abs(snapshots[SNAPSHOTS[-1]] - truth).mean():.3f} rad")

---

## Para profundizar: experimentos de una sola variable

Con el mismo código se puede responder, cambiando **una** línea de
`fresh_config`:

- `use_vae=False`: ¿cambia algo sacar el VAE, si igual colapsa?
- `kl_weight=1` (o 0,1): ¿deja de colapsar la `z`? ¿mejora el error?
- `chunk_size=24` (y `n_action_steps=24`): ¿bloques más cortos aprenden más
  rápido la forma?

Para comparar en serio hay que correr cada variante varias veces (con
distintas semillas) y muchos más pasos: con una corrida corta, la diferencia
puede ser ruido. Los entrenamientos de verdad van a la workstation.